# Get Monthly Density

In [1]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
from glob import glob
from scipy.interpolate import griddata

## Load GLORYS

In [2]:
HOME = '/oscar/data/deeps/private/ediloren/ebeaudin/'
data_path = HOME+'data/GLORYS_daily/depth_0.5m/'
glorys = xr.open_mfdataset(sorted(glob(data_path+'*.nc')))
glorys = glorys.isel(depth=0)

lon, lat= glorys.longitude, glorys.latitude

## Create bin edges using GLORYS

In [3]:
# Bin centers exactly match GLORYS grid
xcenters = glorys.longitude.values
ycenters = glorys.latitude.values

xedges = np.concatenate([
    [xcenters[0] - (xcenters[1] - xcenters[0]) / 2],
    0.5 * (xcenters[:-1] + xcenters[1:]),
    [xcenters[-1] + (xcenters[-1] - xcenters[-2]) / 2]
])
yedges = np.concatenate([
    [ycenters[0] - (ycenters[1] - ycenters[0]) / 2],
    0.5 * (ycenters[:-1] + ycenters[1:]),
    [ycenters[-1] + (ycenters[-1] - ycenters[-2]) / 2]
])

## Load Combined Dataset of Daily Releases

In [5]:
region='Sitka'
depth='47.37m'#0.5m,2.65m,5.08m,9.57m,21.6m,47.37m,92.33m

#file=f'/oscar/data/deeps/private/ediloren/ebeaudin/particle-tracking/{region}_{depth}_1993-2025/all_particles_combined.nc'
file=f'/oscar/data/deeps/private/ediloren/ebeaudin/particle-tracking/{region}_{depth}_LowPass_sigma120_1993-2025/all_particles_combined.nc'
file
ds = xr.open_dataset(file, engine="netcdf4")
times = ds.time
T  = len(ds.time)
ds

FileNotFoundError: [Errno 2] No such file or directory: '/oscar/data/deeps/private/ediloren/ebeaudin/particle-tracking/Sitka_47.37m_LowPass_sigma120_1993-2025/all_particles_combined.nc'

## Function to create 2d histogram

In [5]:
def heatmap(data,xedges, yedges, out=0, plot=1):
    import matplotlib.colors as mcolors
    from matplotlib.colors import LogNorm
    
    dx = 1/2 # physical bin size [°]
    dy = 0.5/2

    cmap = plt.colormaps['inferno'].copy()
    cmap.set_under('black')  # set color for values below vmin
    
    # Flatten the (lag, particle) dims
    x_all = data.lon.values.flatten()
    y_all = data.lat.values.flatten()
    
    # Mask out NaNs
    valid = ~np.isnan(x_all)
    x_valid = x_all[valid]
    y_valid = y_all[valid]

    # data range  
    xmin, xmax = x_valid.min(), x_valid.max()  
    ymin, ymax = y_valid.min(), y_valid.max()
    
    # Plot
    plt.figure(figsize=(6, 4))
    #zos.isel(time=t).plot.contour(colors='k',linewidths=0.8,)
    vmin = 1e-3  # small nonzero to avoid log(0)
    vmax = 10000
    norm = LogNorm(vmin=vmin, vmax=vmax)
    counts, _, _, _ = plt.hist2d(x_valid, y_valid, bins=[xedges, yedges], cmap=cmap, norm=norm)
    #counts, _, _ = np.histogram2d(x_valid, y_valid, bins=[xedges, yedges])
    plt.colorbar(label='Particle Count')
    #plt.title(f'Date: {str(times.isel(time=t).values)[:10]}')
    #coastline()
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.tight_layout()
    if plot==1: plt.show()
    else: plt.close()

    if out==1:
        return counts

## Loop over each month and save monthly density map

In [ ]:
outdir=f'/oscar/data/deeps/private/ediloren/ebeaudin/particle-tracking/{region}_{depth}_LowPass_sigma30_1993-2025/monthly_density/'
os.makedirs(outdir, exist_ok=True)
print(outdir)

# Monthly density maps
for tstart in pd.date_range(ds.time[0].values, ds.time[-1].values, freq='MS'):
    tend = (tstart + pd.DateOffset(months=1)) - pd.Timedelta(seconds=1)
    
    fname = outdir+f"particle_density_{str(tstart)[0:7]}.nc"

    if not os.path.exists(fname):
        
        subset = ds.sel(time=slice(tstart, tend))
        print(str(tstart)[0:7])
        
        counts = heatmap(subset, xedges, yedges, out=1, plot=0)
        
        # Save dataset
        monthly_density = xr.Dataset(
            {
                 "counts": (("lat", "lon",), counts.T),
            },
            coords={
                "lon": glorys.longitude.values,
                "lat": glorys.latitude.values,
                "time": [np.datetime64(tstart, 'ns')],
            },
            attrs={
                "description": f"Particle density map from {tstart} to {tend}",
                "method": 'Runge-Kutta 4th order',
                "depth": depth,
            }
        )

        print(f"Saving {fname}")
        monthly_density.to_netcdf(fname)

/oscar/data/deeps/private/ediloren/ebeaudin/particle-tracking/Sitka_47.37m_LowPass_sigma30_1993-2025/monthly_density/
2003-12
Saving /oscar/data/deeps/private/ediloren/ebeaudin/particle-tracking/Sitka_47.37m_LowPass_sigma30_1993-2025/monthly_density/particle_density_2003-12.nc
2004-01
Saving /oscar/data/deeps/private/ediloren/ebeaudin/particle-tracking/Sitka_47.37m_LowPass_sigma30_1993-2025/monthly_density/particle_density_2004-01.nc
2004-02
Saving /oscar/data/deeps/private/ediloren/ebeaudin/particle-tracking/Sitka_47.37m_LowPass_sigma30_1993-2025/monthly_density/particle_density_2004-02.nc
2004-03
Saving /oscar/data/deeps/private/ediloren/ebeaudin/particle-tracking/Sitka_47.37m_LowPass_sigma30_1993-2025/monthly_density/particle_density_2004-03.nc
2004-04
Saving /oscar/data/deeps/private/ediloren/ebeaudin/particle-tracking/Sitka_47.37m_LowPass_sigma30_1993-2025/monthly_density/particle_density_2004-04.nc
2004-05
Saving /oscar/data/deeps/private/ediloren/ebeaudin/particle-tracking/Sitka

## Only include the 3 years prior

In [ ]:
nyears = 3

outdir=f'/oscar/data/deeps/private/ediloren/ebeaudin/particle-tracking/{region}_{depth}_1993-2025/monthly_densities_{nyears}years/'
os.makedirs(outdir, exist_ok=True)
print(outdir)

for tstart in pd.date_range(ds.time[0].values, ds.time[-1].values, freq='MS'):
    window_start = tstart - pd.Timedelta(days=365*3)  # ~3 years before
    window_end = tstart

    subset = ds.sel(time=slice(window_start, window_end))
    tend = (tstart + pd.DateOffset(months=1)) - pd.Timedelta(seconds=1)

    
    fname = outdir + f"particle_density_{str(tstart)[:7]}.nc"

    if not os.path.exists(fname):
        print(f"Processing for {str(tstart)[:7]} using previous 3 years (~1095 days) from {window_start.date()} to {window_end.date()}")

        subset_month = subset.sel(time=slice(tstart, tend))
        counts = heatmap(subset_month, xedges, yedges, out=1, plot=0)

        monthly_density = xr.Dataset(
            {
                "counts": (("lat", "lon"), counts.T),
            },
            coords={
                "lon": glorys.longitude.values,
                "lat": glorys.latitude.values,
                "time": [np.datetime64(tstart, 'ns')],
            },
            attrs={
                "description": f"Particle density map for {tstart} using previous 3 years (~1095 days) data",
                "method": 'Runge-Kutta 4th order',
                "depth": depth,
            }
        )
        
        monthly_density.to_netcdf(fname)
    

/oscar/data/deeps/private/ediloren/ebeaudin/particle-tracking/Haida_47.37m_1993-2025/monthly_densities_3years/
Processing for 1996-04 using previous 3 years (~1095 days) from 1993-04-02 to 1996-04-01
Processing for 1996-05 using previous 3 years (~1095 days) from 1993-05-02 to 1996-05-01
Processing for 1996-06 using previous 3 years (~1095 days) from 1993-06-02 to 1996-06-01
Processing for 1996-07 using previous 3 years (~1095 days) from 1993-07-02 to 1996-07-01
Processing for 1996-08 using previous 3 years (~1095 days) from 1993-08-02 to 1996-08-01
Processing for 1996-09 using previous 3 years (~1095 days) from 1993-09-02 to 1996-09-01
Processing for 1996-10 using previous 3 years (~1095 days) from 1993-10-02 to 1996-10-01
Processing for 1996-11 using previous 3 years (~1095 days) from 1993-11-02 to 1996-11-01
Processing for 1996-12 using previous 3 years (~1095 days) from 1993-12-02 to 1996-12-01
Processing for 1997-01 using previous 3 years (~1095 days) from 1994-01-02 to 1997-01-01